In [38]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "martin2011memory")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "martinordas2011memory_first experiment.sav")
complete_path_2 = os.path.join(original_data_pathway, "martinordas2011memory_second_experiment.sav")
complete_path_3 = os.path.join(original_data_pathway, "martinordas2011memory_interference_data.sav")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [39]:
import pandas as pd
import numpy as np
import pyreadstat


df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df1 = df1.assign(experiment='1')


remdf=["kuno", "limbuko", "pini",'padana','dokana']
df1 = df1[~df1.subjects.isin(remdf)]
# df1['subjects'].unique()

In [40]:

df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
df2 = df2.assign(experiment='2')

df3 = pd.read_spss(complete_path_3, usecols=None, convert_categoricals=True)
df3 = df3.assign(experiment='3')
df3 = df3.assign(experiment_name='interference')


In [41]:
data_frames=[df1, df2, df3]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subjects": "ape"}, inplace=True)
    x['study_id']="martin2011memory"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [42]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')


In [43]:
fulldf.rename(columns={"filter_$": "filter_dollar"}, inplace=True)
fulldf['ape'].replace('', np.nan, inplace=True)
fulldf.dropna(subset=['ape'], inplace=True)

# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)


fulldf['condition'].replace(' ', '_', inplace=True)
fulldf.dropna(subset=['age_in_years'], inplace=True)

rename_list_columns = [['ni_day','no_interference-day'],
       ['ei_test_day','early_interference-test_day'], 
       ['ei_interference_day','early_interference-interference_day'], 
       ['li_test_day','late_interference-test_day'], 
       ['li_interf_day','late_interference-interference_day'],
       ['ni_night','no_interference-night'], 
       ['ei_test_night','early_interference-test_night'], 
       ['ei_interf_night','early_interference-interference_night'], 
       ['li_test_night','late_interference-test_night'],
       ['li_interf_night','late_interference-interference_night']]

for x,y in rename_list_columns:
    fulldf.rename(columns={x: y}, inplace=True)

# fulldf.columns

In [44]:
fulldf=fulldf[['study_id','experiment', 'participant','age_in_years', 'sex','species',
                # 'condition',
        'two_min', 'one_hour', 'two_hours', 'twentyfour_h', 
        # 'var00001', 
       'four_hours', 'eight_hours', 'twenty_four_hours', 
#        'chance', 
       'no_interference-day', 'early_interference-test_day',
       'early_interference-interference_day', 'late_interference-test_day',
       'late_interference-interference_day', 'no_interference-night',
       'early_interference-test_night',
       'early_interference-interference_night', 'late_interference-test_night',
       'late_interference-interference_night']]


In [45]:
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'martin2011memory_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'martin2011memory_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)